In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer

data = pd.read_csv('train_sessions.csv', index_col=0, parse_dates=['time1', 'time2', 'time3', 'time4', 'time5', 'time6', 'time7', 'time8', 'time9', 'time10'])

data

,site1,time1,site2,time2,site3,time3,site4,time4,site5,time5,...,time6,site7,time7,site8,time8,site9,time9,site10,time10,target
session_id,,,,,,,,,,,,,,,,,,,,,
1,718,2014-02-20 10:02:45,NaN,NaT,NaN,NaT,NaN,NaT,NaN,NaT,...,NaT,NaN,NaT,NaN,NaT,NaN,NaT,NaN,NaT,0
2,890,2014-02-22 11:19:50,941.0,2014-02-22 11:19:50,3847.0,2014-02-22 11:19:51,941.0,2014-02-22 11:19:51,942.0,2014-02-22 11:19:51,...,2014-02-22 11:19:51,3847.0,2014-02-22 11:19:52,3846.0,2014-02-22 11:19:52,1516.0,2014-02-22 11:20:15,1518.0,2014-02-22 11:20:16,0
3,14769,2013-12-16 16:40:17,39.0,2013-12-16 16:40:18,14768.0,2013-12-16 16:40:19,14769.0,2013-12-16 16:40:19,37.0,2013-12-16 16:40:19,...,2013-12-16 16:40:19,14768.0,2013-12-16 16:40:20,14768.0,2013-12-16 16:40:21,14768.0,2013-12-16 16:40:22,14768.0,2013-12-16 16:40:24,0
4,782,2014-03-28 10:52:12,782.0,2014-03-28 10:52:42,782.0,2014-03-28 10:53:12,782.0,2014-03-28 10:53:42,782.0,2014-03-28 10:54:12,...,2014-03-28 10:54:42,782.0,2014-03-28 10:55:12,782.0,2014-03-28 10:55:42,782.0,2014-03-28 10:56:12,782.0,2014-03-28 10:56:42,0
5,22,2014-02-28 10:53:05,177.0,2014-02-28 10:55:22,175.0,2014-02-28 10:55:22,178.0,2014-02-28 10:55:23,177.0,2014-02-28 10:55:23,...,2014-02-28 10:55:59,175.0,2014-02-28 10:55:59,177.0,2014-02-28 10:55:59,177.0,2014-02-28 10:57:06,178.0,2014-02-28 10:57:11,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
253557,3474,2013-11-25 10:26:54,3474.0,2013-11-25 10:26:58,141.0,2013-11-25 10:27:03,2428.0,2013-11-25 10:27:04,106.0,2013-11-25 10:27:13,...,2013-11-25 10:27:16,2428.0,2013-11-25 10:27:28,2428.0,2013-11-25 10:27:40,2428.0,2013-11-25 10:27:52,148.0,2013-11-25 10:27:53,0
253558,12727,2013-03-12 16:01:15,12727.0,2013-03-12 16:01:16,2215.0,2013-03-12 16:01:16,38.0,2013-03-12 16:01:17,2215.0,2013-03-12 16:01:17,...,2013-03-12 16:01:17,25444.0,2013-03-12 16:01:18,2215.0,2013-03-12 16:01:18,23.0,2013-03-12 16:01:18,21.0,2013-03-12 16:01:18,0
253559,2661,2013-09-12 14:05:03,15004.0,2013-09-12 14:05:10,5562.0,2013-09-12 14:05:10,5562.0,2013-09-12 14:06:29,5562.0,2013-09-12 14:06:30,...,NaT,NaN,NaT,NaN,NaT,NaN,NaT,NaN,NaT,0


In [3]:
from sklearn.metrics import roc_auc_score
from lightautoml.tasks import Task
from lightautoml.automl.presets.tabular_presets import TabularAutoML


data = data.fillna(0)

test_data = pd.read_csv('test_sessions.csv', index_col=0)
test_data.fillna(0)
task = Task(name='binary', metric=lambda y_true, y_pred: roc_auc_score(y_true, y_pred))
roles = {'target': 'target'}


auto_ml = TabularAutoML(
    task=task,
    timeout=3600,
    cpu_limit=-1,
    memory_limit=-1,
    general_params={'use_algos': [['linear_l2', 'lgb', 'cb']]},
)

auto_ml.fit_predict(train_data=data, roles=roles)
y_pred = auto_ml.predict(test_data).data

y_pred

array([[2.4616526e-04],
       [2.3510082e-05],
       [3.3958182e-05],
       ...,
       [1.9768546e-03],
       [1.5080579e-04],
       [3.2861702e-05]], dtype=float32)

In [4]:
ans = y_pred.reshape(-1)
pd.Series(ans, index=test_data.index, name="target").to_csv("lightautoml.csv", index=True, header=True, index_label="session_id")

In [32]:
from sklearn.feature_extraction.text import CountVectorizer


data = data.sort_values(by=['time1'])

sites = ['site1', 'site2', 'site3', 'site4', 'site5', 'site6', 'site7', 'site8', 'site9', 'site10']
times = ['time1', 'time2', 'time3', 'time4', 'time5', 'time6', 'time7', 'time8', 'time9', 'time10']
y_train = data.loc[:, 'target'].values
X_train = data.loc[:, sites].fillna(0)
X_test = test_data.loc[:, sites].fillna(0)
cv = CountVectorizer(ngram_range=(1, 3), max_features=50000)

def transform_sites(x: pd.DataFrame, train=False) -> pd:
    x.astype('int').to_csv("temp.txt", sep=' ', index=False, header=False)
    with open('temp.txt', 'r') as f:
        if train:
            return cv.fit_transform(f)
        else:
            return cv.transform(f)

X_train = transform_sites(X_train, train=True)
X_test = transform_sites(X_test)

X_train

<253561x50000 sparse matrix of type '<class 'numpy.int64'>'
	with 3379553 stored elements in Compressed Sparse Row format>

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier


gbc = GradientBoostingClassifier()
gbc.fit(X_train, y_train)

In [41]:
def save_test(estimator, name):
    y_pred = estimator.predict_proba(X_test)[:, 1]
    ans = y_pred.reshape(-1)
    pd.Series(ans, index=test_data.index, name="target").to_csv(name + ".csv", index=True, header=True, index_label="session_id")

In [43]:
from sklearn.linear_model import LogisticRegression


lr = LogisticRegression(solver='liblinear', C=1, random_state=17)
lr.fit(X_train, y_train)

save_test(lr, 'lr')

In [ ]:


def add_time_features(df, X_sparse):
    hour = df['time1'].apply(lambda ts: ts.hour)
    morning = ((hour >= 7) & (hour <= 11)).astype('int')
    day = ((hour >= 12) & (hour <= 18)).astype('int')
    evening = ((hour >= 19) & (hour <= 23)).astype('int')
    night = ((hour >= 0) & (hour <= 6)).astype('int')
    X = np.hstack([X_sparse, morning.values.reshape(-1, 1), 
                day.values.reshape(-1, 1), evening.values.reshape(-1, 1), 
                night.values.reshape(-1, 1)])
    